# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
# TODO
print("Số dòng: ", df.shape[0], "Số cột: ", df.shape[1])
print()
df.info()

Số dòng:  541909 Số cột:  8

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  str           
 1   StockCode    541909 non-null  str           
 2   Description  540455 non-null  str           
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 33.1 MB


## A.2. Missing values & Duplicate data

In [3]:
# TODO
missing = df.isnull().sum()
print("Missing values: ")
print(missing[missing>0] if missing.sum()>0 else "Không có cột nào thiếu dữ liệu.")
print()
print("Duplicate rows (toàn bộ):", df.duplicated().sum())

Missing values: 
Description      1454
CustomerID     135080
dtype: int64

Duplicate rows (toàn bộ): 5268


## A.3. Invalid values

In [4]:
# TODO
print("Quantity <= 0:")
print(df[df["Quantity"] <= 0])

print("UnitPrice <= 0:")
print(df[df["UnitPrice"] <= 0])

Quantity <= 0:
       InvoiceNo StockCode                       Description  Quantity  \
141      C536379         D                          Discount        -1   
154      C536383    35004C   SET OF 3 COLOURED  FLYING DUCKS        -1   
235      C536391     22556    PLASTERS IN TIN CIRCUS PARADE        -12   
236      C536391     21984  PACK OF 12 PINK PAISLEY TISSUES        -24   
237      C536391     21983  PACK OF 12 BLUE PAISLEY TISSUES        -24   
...          ...       ...                               ...       ...   
540449   C581490     23144   ZINC T-LIGHT HOLDER STARS SMALL       -11   
541541   C581499         M                            Manual        -1   
541715   C581568     21258        VICTORIAN SEWING BOX LARGE        -5   
541716   C581569     84978  HANGING HEART JAR T-LIGHT HOLDER        -1   
541717   C581569     20979     36 PENCILS TUBE RED RETROSPOT        -5   

               InvoiceDate  UnitPrice  CustomerID         Country  
141    2010-12-01 09:41:00  

## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [5]:
# TODO
df = df[df["Quantity"] > 0]
df = df[df["UnitPrice"] > 0]
df["Sales"] = df["Quantity"] * df["UnitPrice"]
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
# TODO
print("Mean:")
print(df[["Quantity", "UnitPrice", "Sales"]].mean())

print("\nMedian:")
print(df[["Quantity", "UnitPrice", "Sales"]].median())

print("\nMode:")
print(df[["Quantity", "UnitPrice", "Sales"]].mode())

Mean:
Quantity     10.542037
UnitPrice     3.907625
Sales        20.121871
dtype: float64

Median:
Quantity     3.00
UnitPrice    2.08
Sales        9.90
dtype: float64

Mode:
   Quantity  UnitPrice  Sales
0         1       1.25   15.0


## Group 2 — Dispersion

In [8]:
# TODO
print("Min:")
print(df[["Quantity", "UnitPrice", "Sales"]].min())

print("\nMax:")
print(df[["Quantity", "UnitPrice", "Sales"]].max())

print("\nVariance:")
print(df[["Quantity", "UnitPrice", "Sales"]].var())

print("\nStandard deviation:")
print(df[["Quantity", "UnitPrice", "Sales"]].std())

Min:
Quantity     1.000
UnitPrice    0.001
Sales        0.001
dtype: float64

Max:
Quantity      80995.00
UnitPrice     13541.33
Sales        168469.60
dtype: float64

Variance:
Quantity     24187.752994
UnitPrice     1289.936149
Sales        73092.768604
dtype: float64

Standard deviation:
Quantity     155.524124
UnitPrice     35.915681
Sales        270.356743
dtype: float64


## Group 3 — Location and Shape

In [9]:
# TODO
# B.3

print(df[["Quantity", "UnitPrice", "Sales"]].describe())
print(df[['Quantity', 'UnitPrice', 'Sales']].skew())

            Quantity      UnitPrice          Sales
count  530104.000000  530104.000000  530104.000000
mean       10.542037       3.907625      20.121871
std       155.524124      35.915681     270.356743
min         1.000000       0.001000       0.001000
25%         1.000000       1.250000       3.750000
50%         3.000000       2.080000       9.900000
75%        10.000000       4.130000      17.700000
max     80995.000000   13541.330000  168469.600000
Quantity     471.727716
UnitPrice    206.087555
Sales        506.706012
dtype: float64


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [12]:
# TODO

country_sales = df.groupby("Country")["Sales"].sum()
print(country_sales.sort_values(ascending=False))
print("Quốc gia có doanh thu cao nhất:")
print(country_sales.idxmax())

country_sales = df.groupby("Country")["Sales"].sum()

print(country_sales.idxmax())

print(country_sales.max() / df["Sales"].sum() * 100)

Country
United Kingdom          9025222.084
Netherlands              285446.340
EIRE                     283453.960
Germany                  228867.140
France                   209715.110
Australia                138521.310
Spain                     61577.110
Switzerland               57089.900
Belgium                   41196.340
Sweden                    38378.330
Japan                     37416.370
Norway                    36165.440
Portugal                  33747.100
Finland                   22546.080
Singapore                 21279.290
Channel Islands           20450.440
Denmark                   18955.340
Italy                     17483.240
Hong Kong                 15691.800
Cyprus                    13590.380
Austria                   10198.680
Israel                     8135.260
Poland                     7334.650
Greece                     4760.520
Unspecified                4749.790
Iceland                    4310.000
Canada                     3666.380
USA                 

## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [13]:
# TODO
product_sales = df.groupby("Description")["Sales"].sum()
print(product_sales.sort_values(ascending=False).head())
print(product_sales.idxmax())

Description
DOTCOM POSTAGE                        206248.77
REGENCY CAKESTAND 3 TIER              174484.74
PAPER CRAFT , LITTLE BIRDIE           168469.60
WHITE HANGING HEART T-LIGHT HOLDER    106292.77
PARTY BUNTING                          99504.33
Name: Sales, dtype: float64
DOTCOM POSTAGE


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [14]:
# TODO
df["Month"] = df["InvoiceDate"].dt.month
monthly_sales = df.groupby("Month")["Sales"].sum()
print(monthly_sales)
print(monthly_sales.sort_values(ascending=False))

Month
1      691364.560
2      523631.890
3      717639.360
4      537808.621
5      770536.020
6      761739.900
7      719221.191
8      759138.380
9     1058590.172
10    1154979.300
11    1509496.330
12    1462538.820
Name: Sales, dtype: float64
Month
11    1509496.330
12    1462538.820
10    1154979.300
9     1058590.172
5      770536.020
6      761739.900
8      759138.380
7      719221.191
3      717639.360
1      691364.560
4      537808.621
2      523631.890
Name: Sales, dtype: float64


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [16]:
# TODO
order_sales = df.groupby(["Country", "InvoiceNo"])["Sales"].sum()
aov = order_sales.groupby("Country").mean()
print(aov.sort_values(ascending=False))

print("AOV cao nhất:")
print(aov.idxmax())
print("Giá trị AOV:")
print(aov.max())

Country
Singapore               3039.898571
Netherlands             3036.663191
Australia               2430.198421
Japan                   1969.282632
Lebanon                 1693.880000
Hong Kong               1426.527273
Brazil                  1143.600000
Sweden                  1066.064722
Switzerland             1057.220370
Denmark                 1053.074444
Israel                  1016.907500
Norway                  1004.595556
RSA                     1002.310000
EIRE                     984.215139
Greece                   952.104000
Cyprus                   849.398750
Channel Islands          786.555385
USA                      716.078000
Spain                    684.190111
United Arab Emirates     634.093333
Iceland                  615.714286
Canada                   611.063333
Austria                  599.922353
Portugal                 581.846552
Finland                  549.904390
Malta                    545.118000
France                   534.987526
United Kingdom      

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [18]:
# TODO
return_data = df[df["Quantity"] < 0]
# Do đã xóa Quantity < 0 ở trên nên cần đọc lại dữ liệu gốc để tính tỷ lệ trả hàng
df_original = pd.read_csv(
    csv_path,
    encoding="ISO-8859-1",
    on_bad_lines="skip"
)

return_data = df_original[df_original["Quantity"] < 0]

return_count = return_data.groupby("Country").size()

total_count = df_original.groupby("Country").size()
print(return_count)

return_rate = return_count / total_count * 100
print(return_rate.sort_values(ascending=False))

Country
Australia               74
Austria                  3
Bahrain                  1
Belgium                 38
Channel Islands         10
Cyprus                   8
Czech Republic           5
Denmark                  9
EIRE                   302
European Community       1
Finland                 10
France                 149
Germany                453
Greece                   1
Hong Kong                4
Israel                   2
Italy                   45
Japan                   37
Malta                   15
Netherlands              8
Norway                  14
Poland                  11
Portugal                18
Saudi Arabia             1
Singapore                7
Spain                   48
Sweden                  11
Switzerland             35
USA                    112
United Kingdom        9192
dtype: int64
Country
USA                     38.487973
Czech Republic          16.666667
Malta                   11.811024
Japan                   10.335196
Saudi Arabia            1

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

Doanh thu của cửa hàng tập trung chủ yếu ở một số quốc gia, trong đó United Kingdom đóng góp nhiều nhất. Một số sản phẩm có doanh thu cao hơn hẳn các sản phẩm còn lại. Doanh số giữa các tháng có sự thay đổi nên có thể thấy một phần tính mùa vụ. Giá trị đơn hàng trung bình cũng khác nhau giữa các quốc gia. Ngoài ra, tỷ lệ giao dịch có Quantity âm cũng không giống nhau giữa các nước.